# Task 3 — full-data log-target model (v6)

The v5 log-positive-amount experiment improved MAE on all five paired development folds and the reused holdout. Freeze its fixed gate, amount model, continuous blend, and 80/20 weight, then refit **every fitted component** on all labeled Train rows. The evaluation card records the earlier five-fold procedure estimate; it does not claim a fresh score for the full-data fit. Test is used only for predictions and format checks.

In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.pipeline.build_sample_submission import build_sample_submission
from src.task3.models.spending_models import PositiveGate,PrefitBlend,WeightedPair
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
SRC=ROOT/'src';TASK=SRC/'task3';SAVE=TASK/'save';DATA='eda_combined_v1'
NAME='t3_log_positive_amount_full_v6';SUBMISSION_NAME='t3_log_positive_amount_full_v6'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path'])
test=pd.read_parquet(SRC/card['splits']['test']['path'])
assert train.drop(columns='spending_30d').columns.tolist()==test.columns.tolist()
assert train.player_id.is_unique and test.player_id.is_unique
features=[c for c in test if c!='player_id']
X=train[features];y=train.spending_30d.astype(float)
source=joblib.load(SAVE/'t3_gate_blend_cv5_v2.joblib')
v5=json.loads((SAVE/'t3_log_positive_amount_cv5_v5-evaluation.json').read_text(encoding='utf-8'))
fold_mae=np.asarray(v5['card']['development_fold_MAE_thb'],dtype=float)
assert len(fold_mae)==5 and source.first_weight==.8 and source.first_model.threshold==.5
print('Train',len(train),'Test',len(test),'features',len(features),'CV mean MAE',fold_mae.mean())

Train 104000 Test 25889 features 48 CV mean MAE 2814.9680123616345


## Fit, save, and evaluate the selected full-data model

The four base estimators are cloned from the development model and refitted on all labeled Train. The positive amount estimator is trained only on positive spending rows and learns the log1p target. All preprocessing stays inside fitted pipelines.

In [2]:
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
if not any(p.exists() for p in paths):
    occurrence=clone(source.first_model.occurrence_model)
    occurrence.fit(X,(y>0).astype(int))
    print('Occurrence fitted',flush=True)
    amount=TransformedTargetRegressor(
        regressor=clone(source.first_model.positive_amount_model),
        func=np.log1p,inverse_func=np.expm1,check_inverse=False)
    amount.fit(X.loc[y>0],y.loc[y>0])
    print('Positive amount fitted',flush=True)
    continuous_cat=clone(source.second_model.catboost_model)
    continuous_cat.fit(X,y)
    print('Continuous CatBoost fitted',flush=True)
    continuous_lgb=clone(source.second_model.lightgbm_model)
    continuous_lgb.fit(X,y)
    print('Continuous LightGBM fitted',flush=True)
    model=WeightedPair(PositiveGate(occurrence,amount,.5),
        PrefitBlend(continuous_cat,continuous_lgb,source.second_model.catboost_weight),.8)
    save_model(task=3,name=NAME,model=model,data_name=DATA,
        card={'idea':'Full Train refit of v5 log-positive-amount mixture',
              'source_model':'t3_log_positive_amount_cv5_v5','seed':SEED,
              'rows':len(train),'positive_rows':int((y>0).sum()),
              'gate_threshold':.5,'gate_weight':.8,
              'development_cv_mean_MAE_thb':float(fold_mae.mean())})
    evaluate_model(task=3,name=NAME,score=float(fold_mae.mean()),
        metric_name='MAE_thb_cv5_procedure',higher_is_better=False,
        card={'fold_scores_MAE_thb':fold_mae.tolist(),
              'split':'earlier five development folds; full-data fit not evaluated here',
              'official_NMAE':'unconfirmed'})
else:
    assert all(p.exists() for p in paths),'Inspect incomplete artifact'
    saved=json.loads(paths[1].read_text(encoding='utf-8'))
    assert saved['data_name']==DATA and saved['card']['source_model']=='t3_log_positive_amount_cv5_v5'
    model=joblib.load(paths[0])
print('Saved model:',NAME)

Occurrence fitted


Positive amount fitted


Continuous CatBoost fitted


Continuous LightGBM fitted


Saved model: t3_log_positive_amount_full_v6


## Test prediction and per-task draft

This uses Test only for final prediction and format checks. The generic helper writes a pipe-separated two-column task file; final Kaggle assembly is a separate workflow.

In [3]:
pred=np.asarray(model.predict(test[features]),dtype=float)
assert len(pred)==len(test) and np.isfinite(pred).all() and (pred>=0).all()
assert len(test)==25889
draft=TASK/'submission'/f'{SUBMISSION_NAME}.csv'
draft_card=draft.with_suffix('.json')
if not draft.exists() and not draft_card.exists():
    info=build_sample_submission(task=3,name=SUBMISSION_NAME,model_name=NAME,
        id_column='player_id',
        card={'source_model':NAME,'prediction':'nonnegative THB','purpose':'local task draft'})
else:
    assert draft.exists() and draft_card.exists(),'Inspect incomplete submission draft'
    info=json.loads(draft_card.read_text(encoding='utf-8'))
part=pd.read_csv(draft,sep='|')
assert part.columns.tolist()==['id','task3']
assert part.id.tolist()==test.player_id.tolist()
assert np.allclose(part.task3,pred)
print('Task 3 draft rows:',len(part),'min/median/max:',np.min(pred),np.median(pred),np.max(pred))

Task 3 draft rows: 25889 min/median/max: 0.0 87.05455231682421 272528.8234452313
